Import libraries

In [1]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from cfrnet import CFRnet
import sys
from pathlib import Path
project_root = Path("/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [2]:
%time train_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Women/test_women.csv")
%time val_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Women/val_women.csv")

CPU times: user 25.8 ms, sys: 6.95 ms, total: 32.7 ms
Wall time: 32.1 ms
CPU times: user 12.3 ms, sys: 4.02 ms, total: 16.3 ms
Wall time: 16.3 ms
CPU times: user 4.38 ms, sys: 2.95 ms, total: 7.32 ms
Wall time: 7.33 ms


In [3]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['conversion']
treatment_feature = ['treatment']

In [4]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [5]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [6]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [7]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4270, 10]); y=torch.Size([4270, 1]); t=torch.Size([4270, 1])
Shape of test: x=torch.Size([12808, 10]); y=torch.Size([12808, 1]); t=torch.Size([12808, 1])


In [8]:
epochs = 150
alpha = 1
lr = 1e-3
wd = 1e-5
method = "mmd_rbf"
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
early_stop_start = 0
shared_hidden = 200
outcome_hidden = 100
outcome_dropout = 0.0
shared_dropout = 0.0 
activation = torch.nn.ReLU

print (f" epochs = {epochs}")
print (f" method = {method}")
print (f" alpha = {alpha}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f"activation = {activation}")

 epochs = 150
 method = mmd_rbf
 alpha = 1
 learning rate = 0.001
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
activation = <class 'torch.nn.modules.activation.ReLU'>


In [9]:
import pandas as pd
import numpy as np
import torch
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-5, 1e-4, log=True)
    grid_alpha = trial.suggest_categorical("alpha", [0.1, 0.5, 1.0])
    grid_method = trial.suggest_categorical("method", ["mmd_rbf", "mmd_linear"])
    val_scores = []
    for SEED in seeds:
        seed_everything(SEED)

        cfr = CFRnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            alpha=grid_alpha,
            method=grid_method,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_dropout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        # Silence CFRNet epoch logs
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            cfr.fit(train_loader, val_loader)

        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p= cfr.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)
        val_scores.append(current_val_auqc)

    mean_val_auqc = float(np.mean(val_scores))
    std_val_auqc = float(np.std(val_scores, ddof=1)) if len(val_scores) > 1 else 0.0
    trial.set_user_attr("std_Val_AUQC", std_val_auqc)
    return mean_val_auqc

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

# 2. Persist search results in notebook variables
trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": t.params["lr"],
        "weight_decay": t.params["weight_decay"],
        "alpha": t.params["alpha"],
        "method": t.params["method"],
        "mean_Val_AUQC": float(t.value),
        "std_Val_AUQC": float(t.user_attrs.get("std_Val_AUQC", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_Val_AUQC", ascending=False).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "alpha": float(best_params["alpha"]),
    "method": t.params["method"],
    "mean_Val_AUQC": float(study.best_value),
    "std_Val_AUQC": float(study.best_trial.user_attrs.get("std_Val_AUQC", np.nan))
})

/home/ducvu0904/miniconda3/envs/ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/50 [00:00<?, ?it/s]

Locked random seed: 412312


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 42


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 902745


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 1


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
Best trial: 21. Best value: 0.852856: 100%|██████████| 50/50 [44:36<00:00, 53.53s/it]


In [10]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_AUQC: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 21
Best mean_Val_AUQC: 0.852856
Best params: {'lr': 0.00011327625005423851, 'weight_decay': 2.1878042054849755e-05, 'alpha': 0.5, 'method': 'mmd_linear'}

Best config table:


,lr,weight_decay,alpha,method,mean_Val_AUQC,std_Val_AUQC
0,0.000113,0.000022,0.5,mmd_linear,0.852856,0.076983



Top 10 trials:


,trial,lr,weight_decay,alpha,method,mean_Val_AUQC,std_Val_AUQC
0,21,0.000113,0.000022,0.5,mmd_linear,0.852856,0.076983
1,19,0.000112,0.000033,0.5,mmd_linear,0.852278,0.076960
2,16,0.000121,0.000031,0.1,mmd_linear,0.851364,0.067761
3,18,0.000120,0.000020,0.5,mmd_rbf,0.850125,0.065760
4,37,0.000112,0.000025,0.5,mmd_rbf,0.850004,0.065797
5,7,0.000160,0.000017,0.1,mmd_rbf,0.849207,0.062809
6,6,0.000146,0.000090,0.1,mmd_linear,0.849200,0.071993
7,12,0.000107,0.000096,0.1,mmd_rbf,0.849023,0.057570
8,40,0.000162,0.000018,0.5,mmd_rbf,0.848998,0.065112
9,35,0.000179,0.000036,0.5,mmd_linear,0.848672,0.068580


In [13]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    cfrnet = CFRnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=1.13E-04, 
        alpha=0.5, method="mmd_linear",
        weight_decay=2.20E-05, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    cfrnet.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = cfrnet.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 1
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/150 | Loss: 0.5045 | Val Loss: 0.5028 | Val Qini: 0.7268 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7268 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4972 | Val Loss: 0.4955 | Val Qini: 0.7408 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7289 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.4904 | Val Loss: 0.4889 | Val Qini: 0.8022 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7399 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.4840 | Val Loss: 0.4825 | Val Qini: 0.6705 (patience: 1/20)EMA Trend: 0.7295 | (patience: 1/20)
Epoch 5/150 | Loss: 0.4776 | Val Loss: 0.4758 | Val Qini: 0.6198 (patience: 2/20)EMA Trend: 0.7130 | (patience: 2/20)
Epoch

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5182 | Val Loss: 0.5165 | Val Qini: 0.1239 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.1239 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.5110 | Val Loss: 0.5092 | Val Qini: -0.0727 (patience: 1/20)EMA Trend: 0.0944 | (patience: 1/20)
Epoch 3/150 | Loss: 0.5037 | Val Loss: 0.5016 | Val Qini: -0.1411 (patience: 2/20)EMA Trend: 0.0591 | (patience: 2/20)
Epoch 4/150 | Loss: 0.4955 | Val Loss: 0.4931 | Val Qini: -0.1737 (patience: 3/20)EMA Trend: 0.0242 | (patience: 3/20)
Epoch 5/150 | Loss: 0.4858 | Val Loss: 0.4829 | Val Qini: -0.1029 (patience: 4/20)EMA Trend: 0.0051 | (patience: 4/20)
Epoch 6/150 | Loss: 0.4737 | Val Loss: 0.4704 | Val Qini: -0.0736 (patience: 5/20)EMA Trend: -0.0067 | (patience: 5/20)
Epoch 7/150 | Loss: 0.4592 | Val Loss: 0.4550 | Val Qini: -0.0393 (patience: 6/20)EMA Trend: -0.0116 | (patience: 6/20)
Epoch 8/150 | Loss: 0.4410 | Val Loss: 0.4355 | Val Qini: 0.0662 ✓ above trend but not peak (patience: 7/20)EMA Trend: 0.0001 | ✓ above trend but no

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.4961 | Val Loss: 0.4938 | Val Qini: 0.2911 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.2911 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4880 | Val Loss: 0.4855 | Val Qini: 0.3260 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.2964 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.4790 | Val Loss: 0.4766 | Val Qini: 0.5349 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3322 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.4694 | Val Loss: 0.4668 | Val Qini: 0.7267 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3913 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.4592 | Val Loss: 0.4555 | Val Qini: 0.7557 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4460 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 0.4460 | Val Loss: 0.4417 | Val Qini: 0.7405 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.4902 | ✓ above trend but not peak (patience: 1/20)
Epoch 7/150 | Loss: 0.4296 | Val Loss: 0.4243 | Val Qini: 0.7766 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5331 | ⭐ NEW BEST (peak ≥ trend)
Epoch 8/150 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5096 | Val Loss: 0.5071 | Val Qini: 0.8809 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.8809 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4997 | Val Loss: 0.4971 | Val Qini: 0.8915 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.8825 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.4892 | Val Loss: 0.4863 | Val Qini: 0.8350 (patience: 1/20)EMA Trend: 0.8753 | (patience: 1/20)
Epoch 4/150 | Loss: 0.4778 | Val Loss: 0.4744 | Val Qini: 0.7518 (patience: 2/20)EMA Trend: 0.8568 | (patience: 2/20)
Epoch 5/150 | Loss: 0.4645 | Val Loss: 0.4602 | Val Qini: 0.7816 (patience: 3/20)EMA Trend: 0.8455 | (patience: 3/20)
Epoch 6/150 | Loss: 0.4479 | Val Loss: 0.4427 | Val Qini: 0.8207 (patience: 4/20)EMA Trend: 0.8418 | (patience: 4/20)
Epoch 7/150 | Loss: 0.4270 | Val Loss: 0.4210 | Val Qini: 0.8379 (patience: 5/20)EMA Trend: 0.8412 | (patience: 5/20)
Epoch 8/150 | Loss: 0.4027 | Val Loss: 0.3946 | Val Qini: 0.7849 (patience: 6/20)EMA Trend: 0.8328 | (patience: 6/20)
Epoch 9/150 | Loss: 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5066 | Val Loss: 0.5051 | Val Qini: 0.7054 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7054 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4998 | Val Loss: 0.4982 | Val Qini: 0.7477 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7117 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.4927 | Val Loss: 0.4907 | Val Qini: 0.7070 (patience: 1/20)EMA Trend: 0.7110 | (patience: 1/20)
Epoch 4/150 | Loss: 0.4842 | Val Loss: 0.4819 | Val Qini: 0.6916 (patience: 2/20)EMA Trend: 0.7081 | (patience: 2/20)
Epoch 5/150 | Loss: 0.4742 | Val Loss: 0.4713 | Val Qini: 0.7523 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7147 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 0.4622 | Val Loss: 0.4582 | Val Qini: 0.7554 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7208 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 0.4469 | Val Loss: 0.4422 | Val Qini: 0.7213 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.7209 | ✓ above trend but not peak (patience: 1/20)
Epoch 8/150 | Loss: 0.4279 | Val Loss: 0.4224 | 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


In [14]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [4212312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    cfrnet = CFRnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=0.000158, 
        alpha=1, method="mmd_rbf",
        weight_decay=0.000021, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    cfrnet.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = cfrnet.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 4212312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 1
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/150 | Loss: 0.4937 | Val Loss: 0.4914 | Val Qini: 1.0413 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.0413 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4846 | Val Loss: 0.4819 | Val Qini: 1.1402 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.0561 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.4741 | Val Loss: 0.4710 | Val Qini: 1.0283 (patience: 1/20)EMA Trend: 1.0519 | (patience: 1/20)
Epoch 4/150 | Loss: 0.4615 | Val Loss: 0.4572 | Val Qini: 0.9413 (patience: 2/20)EMA Trend: 1.0353 | (patience: 2/20)
Epoch 5/150 | Loss: 0.4446 | Val Loss: 0.4388 | Val Qini: 0.9158 (patience: 3/20)EMA Trend: 1.0174 | (patience: 3/20)
Epoch 6/150 | Loss: 0.

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5160 | Val Loss: 0.5134 | Val Qini: 0.0432 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.0432 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.5059 | Val Loss: 0.5032 | Val Qini: -0.1289 (patience: 1/20)EMA Trend: 0.0174 | (patience: 1/20)
Epoch 3/150 | Loss: 0.4951 | Val Loss: 0.4916 | Val Qini: -0.1698 (patience: 2/20)EMA Trend: -0.0107 | (patience: 2/20)
Epoch 4/150 | Loss: 0.4815 | Val Loss: 0.4773 | Val Qini: -0.1088 (patience: 3/20)EMA Trend: -0.0254 | (patience: 3/20)
Epoch 5/150 | Loss: 0.4644 | Val Loss: 0.4588 | Val Qini: -0.0456 (patience: 4/20)EMA Trend: -0.0284 | (patience: 4/20)
Epoch 6/150 | Loss: 0.4413 | Val Loss: 0.4344 | Val Qini: 0.0512 ⭐ NEW BEST (peak ≥ trend)EMA Trend: -0.0165 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 0.4105 | Val Loss: 0.4009 | Val Qini: 0.1844 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.0136 | ⭐ NEW BEST (peak ≥ trend)
Epoch 8/150 | Loss: 0.3688 | Val Loss: 0.3559 | Val Qini: 0.2154 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.0439 | ⭐ 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.4936 | Val Loss: 0.4904 | Val Qini: 0.3094 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3094 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4820 | Val Loss: 0.4784 | Val Qini: 0.4704 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3336 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.4689 | Val Loss: 0.4652 | Val Qini: 0.7329 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3935 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.4538 | Val Loss: 0.4494 | Val Qini: 0.7245 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.4431 | ✓ above trend but not peak (patience: 1/20)
Epoch 5/150 | Loss: 0.4352 | Val Loss: 0.4286 | Val Qini: 0.7166 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.4842 | ✓ above trend but not peak (patience: 2/20)
Epoch 6/150 | Loss: 0.4093 | Val Loss: 0.4007 | Val Qini: 0.8135 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5336 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 0.3744 | Val Loss: 0.3631 | Val Qini: 0.8597 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5825 | ⭐ 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5066 | Val Loss: 0.5030 | Val Qini: 0.8864 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.8864 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4925 | Val Loss: 0.4885 | Val Qini: 0.8610 (patience: 1/20)EMA Trend: 0.8826 | (patience: 1/20)
Epoch 3/150 | Loss: 0.4770 | Val Loss: 0.4724 | Val Qini: 0.7511 (patience: 2/20)EMA Trend: 0.8629 | (patience: 2/20)
Epoch 4/150 | Loss: 0.4583 | Val Loss: 0.4524 | Val Qini: 0.7956 (patience: 3/20)EMA Trend: 0.8528 | (patience: 3/20)
Epoch 5/150 | Loss: 0.4342 | Val Loss: 0.4263 | Val Qini: 0.8429 (patience: 4/20)EMA Trend: 0.8513 | (patience: 4/20)
Epoch 6/150 | Loss: 0.4026 | Val Loss: 0.3927 | Val Qini: 0.7912 (patience: 5/20)EMA Trend: 0.8423 | (patience: 5/20)
Epoch 7/150 | Loss: 0.3626 | Val Loss: 0.3503 | Val Qini: 0.7787 (patience: 6/20)EMA Trend: 0.8328 | (patience: 6/20)
Epoch 8/150 | Loss: 0.3126 | Val Loss: 0.2964 | Val Qini: 0.7570 (patience: 7/20)EMA Trend: 0.8214 | (patience: 7/20)
Epoch 9/150 | Loss: 0.2508 | Val Loss:

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5044 | Val Loss: 0.5022 | Val Qini: 0.7333 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7333 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4947 | Val Loss: 0.4922 | Val Qini: 0.7180 (patience: 1/20)EMA Trend: 0.7310 | (patience: 1/20)
Epoch 3/150 | Loss: 0.4837 | Val Loss: 0.4803 | Val Qini: 0.6996 (patience: 2/20)EMA Trend: 0.7263 | (patience: 2/20)
Epoch 4/150 | Loss: 0.4696 | Val Loss: 0.4653 | Val Qini: 0.7779 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7340 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.4515 | Val Loss: 0.4460 | Val Qini: 0.7444 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.7356 | ✓ above trend but not peak (patience: 1/20)
Epoch 6/150 | Loss: 0.4291 | Val Loss: 0.4210 | Val Qini: 0.6924 (patience: 2/20)EMA Trend: 0.7291 | (patience: 2/20)
Epoch 7/150 | Loss: 0.3975 | Val Loss: 0.3873 | Val Qini: 0.4333 (patience: 3/20)EMA Trend: 0.6848 | (patience: 3/20)
Epoch 8/150 | Loss: 0.3545 | Val Loss: 0.3411 | Val Qini: 0.2665 (patience: 4/20)EMA

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
